In [0]:
from pyspark.sql import functions as F
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
from pyspark.ml.functions import vector_to_array
from pyspark.sql import functions as F


In [0]:
from pyspark.sql import functions as F

In [0]:
spark.sql("USE CATALOG workspace")
spark.sql("USE SCHEMA health")


# Đọc bảng Gold (chỉnh lại tên nếu bạn dùng tên khác)
df = spark.table("gold_mart_patient_day")

df.printSchema()
df.select("patient_id", "date", "risk_label").show(5)

In [0]:
label_col = "risk_label"

# Bộ features CLEAN (không dùng các cột flag)
numeric_cols_clean = [
    "avg_hr", "avg_sys", "avg_dia", "avg_spo2",
    "cholesterol", "glucose", "hemoglobin", "age_est"
]
categorical_cols_clean = ["gender", "region"]

# Chia train–test
train_df, test_df = df.randomSplit([0.8, 0.2], seed=42)
print("Train:", train_df.count(), "rows")
print("Test :", test_df.count(), "rows")

In [0]:
label_indexer_clean = StringIndexer(
    inputCol=label_col,
    outputCol="label",
    handleInvalid="keep"
)

In [0]:
cat_indexers_clean = [
    StringIndexer(inputCol=c, outputCol=f"{c}_idx", handleInvalid="keep")
    for c in categorical_cols_clean
]
cat_encoder_clean = OneHotEncoder(
    inputCols=[f"{c}_idx" for c in categorical_cols_clean],
    outputCols=[f"{c}_oh"  for c in categorical_cols_clean],
    handleInvalid="keep"
)

In [0]:
feature_cols_clean = numeric_cols_clean + [f"{c}_oh" for c in categorical_cols_clean]

assembler_rf_clean = VectorAssembler(
    inputCols=feature_cols_clean,
    outputCol="features_rf"
)

In [0]:
rf_clean = RandomForestClassifier(
    featuresCol="features_rf",
    labelCol="label",
    numTrees=100,
    maxDepth=8,
    featureSubsetStrategy="auto",
    seed=42
)

pipeline_rf_clean = Pipeline(
    stages=[label_indexer_clean] + cat_indexers_clean +
           [cat_encoder_clean, assembler_rf_clean, rf_clean]
)

rf_clean_model = pipeline_rf_clean.fit(train_df)

In [0]:
pred_test = rf_clean_model.transform(test_df)

evaluator = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction")

acc = evaluator.setMetricName("accuracy").evaluate(pred_test)
f1  = evaluator.setMetricName("f1").evaluate(pred_test)

print("Accuracy (test):", acc)
print("F1        (test):", f1)


In [0]:
label_indexer_stage = rf_clean_model.stages[0]
id2label_clean = {i: lab for i, lab in enumerate(label_indexer_stage.labels)}
print("Label index mapping:", id2label_clean)


pred_rf_all = rf_clean_model.transform(df)

pred_rf_all.select(
    "patient_id", "date", "risk_label", "prediction", "probability"
).show(5, truncate=False)

In [0]:
# Tìm index cho từng class
idx_normal   = [i for i, lab in id2label_clean.items() if lab.lower() == "normal"][0]
idx_elevated = [i for i, lab in id2label_clean.items() if lab.lower() == "elevated"][0]
idx_high     = [i for i, lab in id2label_clean.items() if lab.lower() == "high"][0]

print("Class indices:", idx_normal, idx_elevated, idx_high)



In [0]:
@F.udf("string")
def id_to_label_clean(i):
    return id2label_clean.get(int(i), "UNKNOWN")

# 1) Convert probability (Vector) -> array<double>
pred_rf_with_arr = pred_rf_all.withColumn(
    "prob_arr",
    vector_to_array("probability")
)

# 2) Tạo các cột prob_* và risk_score
pred_rf_serving = (
    pred_rf_with_arr
    .withColumn("prob_normal",   F.col("prob_arr")[idx_normal])
    .withColumn("prob_elevated", F.col("prob_arr")[idx_elevated])
    .withColumn("prob_high",     F.col("prob_arr")[idx_high])
    .withColumn("risk_score",    F.col("prob_elevated") + F.col("prob_high"))
    .withColumn("risk_label_pred", id_to_label_clean("prediction"))
    # nếu muốn gọn, có thể .drop("prob_arr") ở cuối
)

In [0]:
pred_rf_serving.select(
    "patient_id",
    "date",
    "risk_label",
    "risk_label_pred",
    "prob_normal",
    "prob_elevated",
    "prob_high",
    "risk_score"
).show(10)

In [0]:
serving_cols = [
    "patient_id",
    "date",
    "risk_label",       # ground truth
    "risk_label_pred",
    "prob_normal",
    "prob_elevated",
    "prob_high",
    "risk_score",
    "avg_hr",
    "avg_sys",
    "avg_dia",
    "avg_spo2",
    "cholesterol",
    "glucose",
    "hemoglobin",
    "age_est",
    "gender",
    "region"
]

serving_df = pred_rf_serving.select(*serving_cols)

serving_df.show(10)

In [0]:
serving_df.write.mode("overwrite").format("delta").saveAsTable("gold_serving_risk_label")


In [0]:
spark.table("gold_serving_risk_label").show(10)
spark.table("gold_serving_risk_label").count()


In [0]:
# Đăng ký temp view từ serving_df
serving_df.createOrReplaceTempView("serving_temp")

# Tạo bảng Delta từ temp view
spark.sql("""
CREATE OR REPLACE TABLE gold_serving_risk_label
USING DELTA
AS
SELECT * FROM serving_temp
""")


In [0]:
spark.sql("""
SELECT risk_label, risk_label_pred, COUNT(*) AS cnt
FROM gold_serving_risk_label
GROUP BY risk_label, risk_label_pred
ORDER BY risk_label, risk_label_pred
""").show()


In [0]:
spark.sql("""
SELECT patient_id, date, risk_label, risk_label_pred,
       risk_score, prob_high, prob_elevated, avg_sys, avg_dia, glucose, cholesterol
FROM gold_serving_risk_label
ORDER BY risk_score DESC
LIMIT 20
""").show(truncate=False)


In [0]:
spark.sql("""
SELECT region, risk_label_pred, COUNT(*) AS cnt
FROM gold_serving_risk_label
GROUP BY region, risk_label_pred
ORDER BY region, risk_label_pred
""").show()


In [0]:
spark.sql("SELECT current_catalog() AS catalog, current_database() AS db").show()


In [0]:
spark.sql("SHOW TABLES").show(truncate=False)

In [0]:
serving_df = spark.table("gold_serving_risk_label")

serving_df_enriched = (serving_df
    .withColumn("date_col", F.to_date("date"))   # nếu date đã là DateType thì đổi thành F.col("date")
    .withColumn("year",  F.year("date_col"))
    .withColumn("month", F.month("date_col"))
    .withColumn("day",   F.dayofmonth("date_col"))
    .drop("date")
    .withColumnRenamed("date_col", "date")
)

(serving_df_enriched.write
    .mode("overwrite")
    .format("delta")
    .option("overwriteSchema", "true")   # <-- quan trọng
    .saveAsTable("gold_serving_risk_label")
)

spark.table("gold_serving_risk_label").show(5)